In [0]:
class batchWC():

    def __init__(self):
        self.vol_path='/Volumes/cat_spark_streaming/wordcnt/wordcnt/'

    def getRawData(self):
        from pyspark.sql.functions import explode,split,trim,lower,col,count
        df=(spark.read 
                .format('text') 
                .option('multiLine', True) 
                .option('lineSep', True) 
                .load(self.vol_path))
        raw_words=df.select(explode(split(col('value')," ")).alias('word'))
        return raw_words
    
    def getQualityData(self, df):
        return (df.select(trim(col('word')).alias('word')) 
                                .where("word is not null") 
                                .where("word rlike '[a-z]'"))

    def getWordCount(self,df):
        return (df 
                .groupBy('word') 
                .agg(count(col('word'))))
        
    def overWriteWordCount(self,df):
        (df.write
            .format('delta')
            .mode('overwrite')
            .saveAsTable(word_cnt_batch_table))
    
    def wordCount(self):
        print('Starting Batch Execution...',end=' ')
        rawDf=self.getRawData()
        qualifiedDf=self.getQualityData(rawDf)
        wordCountDf=self.getWordCount(qualifiedDf)
        self.overWriteWordCount(wordCountDf)
        print('Completed')
